# Informatica XML Folder Comparison Notebook

Compares two folders containing Informatica PowerCenter XML exports and reports file, object, attribute, and value-level changes.

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import defaultdict
from dataclasses import dataclass, asdict
from typing import Dict, Optional, Tuple
import hashlib
import json
import pandas as pd

## 1. Configuration

In [ ]:
OLD_FOLDER = r"/path/to/informatica_old"
NEW_FOLDER = r"/path/to/informatica_new"
OUTPUT_FOLDER = r"/path/to/informatica_compare_report"

IGNORE_ATTRIBUTES = {
    # "TIMESTAMP",
    # "VERSIONNUMBER",
}
IGNORE_TEXT = False
CASE_INSENSITIVE_FILE_MATCH = False

## 2. Comparator Functions

In [ ]:
IDENTITY_GROUPS = [
    ("NAME",),
    ("FROMINSTANCE", "FROMFIELD", "TOINSTANCE", "TOFIELD"),
    ("FROMINSTANCETYPE", "FROMINSTANCE", "FROMFIELD", "TOINSTANCETYPE", "TOINSTANCE", "TOFIELD"),
    ("INSTANCE_NAME",), ("INSTANCENAME",), ("FIELDNAME",), ("ATTRNAME",),
    ("TASKNAME",), ("SESSIONNAME",), ("WORKFLOWNAME",), ("REFOBJECTNAME",),
    ("DBDNAME",), ("SOURCENAME",), ("TARGETNAME",), ("TRANSFORMATIONNAME",),
    ("MAPPINGNAME",), ("WIDGETTYPE", "NAME"),
]

@dataclass
class DiffRecord:
    file: str
    change_type: str
    object_type: str
    object_path: str
    attribute: str = ""
    old_value: str = ""
    new_value: str = ""
    details: str = ""

def strip_namespace(tag):
    return tag.split("}",1)[1] if "}" in tag else tag

def clean_text(v):
    return "" if v is None else " ".join(v.split())

def norm(v):
    return "" if v is None else v.strip()

def element_identity(elem):
    tag = strip_namespace(elem.tag); attrs = elem.attrib
    for group in IDENTITY_GROUPS:
        if all(k in attrs for k in group):
            return tag, tuple((k, norm(attrs.get(k))) for k in group)
    if "NAME" in attrs:
        keys=["NAME"]
        for x in ("TYPE","OBJECTTYPE","TRANSFORMATIONTYPE","INSTANCETYPE"):
            if x in attrs: keys.append(x)
        return tag, tuple((k,norm(attrs.get(k))) for k in keys)
    preferred=[]
    for k in sorted(attrs):
        if k.upper() in {"TYPE","OBJECTTYPE","TRANSFORMATIONTYPE","INSTANCE_TYPE","DATATYPE","PORTTYPE","PRECISION","SCALE"}:
            preferred.append((k,norm(attrs[k])))
    return tag, tuple(preferred)

def identity_label(elem):
    tag, parts = element_identity(elem)
    return f"{tag}[" + ", ".join(f"{k}={v}" for k,v in parts) + "]" if parts else tag

def child_buckets(parent):
    d=defaultdict(list)
    for c in list(parent): d[element_identity(c)].append(c)
    return d

def attrs_for_compare(elem, ignore):
    return {k:norm(v) for k,v in elem.attrib.items() if k.upper() not in ignore}

def snapshot(elem, ignore):
    return json.dumps({"tag":strip_namespace(elem.tag),"attrs":attrs_for_compare(elem,ignore),"text":clean_text(elem.text)}, sort_keys=True)

def add_descendants(elem,file_name,path,diffs,change_type,ignore):
    for child in list(elem):
        p=f"{path}/{identity_label(child)}"; snap=snapshot(child,ignore)
        diffs.append(DiffRecord(file_name,change_type,strip_namespace(child.tag),p,old_value=snap if change_type=="OBJECT_REMOVED" else "",new_value=snap if change_type=="OBJECT_ADDED" else "",details="Descendant of added/removed object"))
        add_descendants(child,file_name,p,diffs,change_type,ignore)

def compare_elements(old_elem,new_elem,file_name,path,diffs,ignore,ignore_text):
    ot,nt=strip_namespace(old_elem.tag),strip_namespace(new_elem.tag)
    if ot!=nt:
        diffs.append(DiffRecord(file_name,"OBJECT_TYPE_CHANGED",f"{ot}->{nt}",path,old_value=ot,new_value=nt)); return
    oa,na=attrs_for_compare(old_elem,ignore),attrs_for_compare(new_elem,ignore)
    for a in sorted(set(oa)|set(na)):
        if a not in oa: diffs.append(DiffRecord(file_name,"ATTRIBUTE_ADDED",ot,path,a,"",na[a]))
        elif a not in na: diffs.append(DiffRecord(file_name,"ATTRIBUTE_REMOVED",ot,path,a,oa[a],""))
        elif oa[a]!=na[a]: diffs.append(DiffRecord(file_name,"ATTRIBUTE_CHANGED",ot,path,a,oa[a],na[a]))
    if not ignore_text:
        x,y=clean_text(old_elem.text),clean_text(new_elem.text)
        if x!=y: diffs.append(DiffRecord(file_name,"TEXT_CHANGED",ot,path,"#text",x,y))
    ob,nb=child_buckets(old_elem),child_buckets(new_elem)
    for key in sorted(set(ob)|set(nb), key=lambda x:(x[0],str(x[1]))):
        ol,nl=ob.get(key,[]),nb.get(key,[]); m=max(len(ol),len(nl))
        for i in range(m):
            oc=ol[i] if i<len(ol) else None; nc=nl[i] if i<len(nl) else None
            ref=oc if oc is not None else nc; suffix=f"#{i+1}" if m>1 else ""; p=f"{path}/{identity_label(ref)}{suffix}"
            if oc is None:
                diffs.append(DiffRecord(file_name,"OBJECT_ADDED",strip_namespace(nc.tag),p,new_value=snapshot(nc,ignore),details="Object exists only in NEW folder")); add_descendants(nc,file_name,p,diffs,"OBJECT_ADDED",ignore)
            elif nc is None:
                diffs.append(DiffRecord(file_name,"OBJECT_REMOVED",strip_namespace(oc.tag),p,old_value=snapshot(oc,ignore),details="Object exists only in OLD folder")); add_descendants(oc,file_name,p,diffs,"OBJECT_REMOVED",ignore)
            else: compare_elements(oc,nc,file_name,p,diffs,ignore,ignore_text)

def list_xml_files(folder, case_insensitive=False):
    out={}
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower()==".xml":
            rel=p.relative_to(folder).as_posix(); out[rel.lower() if case_insensitive else rel]=p
    return out


## 3. Run Comparison

In [ ]:
old_folder=Path(OLD_FOLDER).expanduser(); new_folder=Path(NEW_FOLDER).expanduser(); output_folder=Path(OUTPUT_FOLDER).expanduser()
if not old_folder.is_dir(): raise FileNotFoundError(f"OLD_FOLDER does not exist: {old_folder}")
if not new_folder.is_dir(): raise FileNotFoundError(f"NEW_FOLDER does not exist: {new_folder}")
output_folder.mkdir(parents=True,exist_ok=True)
ignore={x.upper() for x in IGNORE_ATTRIBUTES}
old_files=list_xml_files(old_folder,CASE_INSENSITIVE_FILE_MATCH); new_files=list_xml_files(new_folder,CASE_INSENSITIVE_FILE_MATCH)
all_keys=sorted(set(old_files)|set(new_files)); file_summary=[]; all_diffs=[]
for key in all_keys:
    op,np=old_files.get(key),new_files.get(key)
    name=(op.relative_to(old_folder).as_posix() if op else np.relative_to(new_folder).as_posix())
    if op is None:
        all_diffs.append(DiffRecord(name,"FILE_ADDED","FILE",name,new_value=str(np),details="XML file exists only in NEW folder")); file_summary.append({"file":name,"status":"FILE_ADDED","change_count":1,"error":""}); continue
    if np is None:
        all_diffs.append(DiffRecord(name,"FILE_REMOVED","FILE",name,old_value=str(op),details="XML file exists only in OLD folder")); file_summary.append({"file":name,"status":"FILE_REMOVED","change_count":1,"error":""}); continue
    try:
        oroot=ET.parse(op).getroot(); nroot=ET.parse(np).getroot(); fd=[]
        compare_elements(oroot,nroot,name,identity_label(oroot),fd,ignore,IGNORE_TEXT); all_diffs.extend(fd)
        file_summary.append({"file":name,"status":"UNCHANGED" if not fd else "CHANGED","change_count":len(fd),"error":""})
    except Exception as e:
        file_summary.append({"file":name,"status":"ERROR","change_count":0,"error":str(e)}); all_diffs.append(DiffRecord(name,"COMPARE_ERROR","FILE",name,details=str(e)))
print(f"Compared {len(all_keys):,} XML files")
print(f"Detailed changes: {len(all_diffs):,}")

## 4. Detailed Changes

In [ ]:
detail_columns=["file","change_type","object_type","object_path","attribute","old_value","new_value","details"]
detail_df=pd.DataFrame([asdict(x) for x in all_diffs],columns=detail_columns) if all_diffs else pd.DataFrame(columns=detail_columns)
detail_df

## 5. File Summary

In [ ]:
file_summary_df=pd.DataFrame(file_summary)
file_summary_df

## 6. Change Type Summary

In [ ]:
change_summary_df=(detail_df.groupby("change_type").size().reset_index(name="count").sort_values("count",ascending=False)) if not detail_df.empty else pd.DataFrame(columns=["change_type","count"])
change_summary_df

## 7. Object Type Summary

In [ ]:
object_summary_df=(detail_df.groupby("object_type").size().reset_index(name="count").sort_values("count",ascending=False)) if not detail_df.empty else pd.DataFrame(columns=["object_type","count"])
object_summary_df

## 8. Useful Filters

In [ ]:
attribute_changes_df=detail_df[detail_df["change_type"].isin(["ATTRIBUTE_CHANGED","ATTRIBUTE_ADDED","ATTRIBUTE_REMOVED"])].copy()
attribute_changes_df

In [ ]:
object_changes_df=detail_df[detail_df["change_type"].isin(["OBJECT_ADDED","OBJECT_REMOVED"])].copy()
object_changes_df

In [ ]:
connector_changes_df=detail_df[detail_df["object_type"].astype(str).str.upper().eq("CONNECTOR")].copy()
connector_changes_df

## 9. Export Reports

In [ ]:
file_summary_df.to_csv(output_folder/"informatica_xml_file_summary.csv",index=False,encoding="utf-8-sig")
detail_df.to_csv(output_folder/"informatica_xml_detailed_changes.csv",index=False,encoding="utf-8-sig")
change_summary_df.to_csv(output_folder/"informatica_xml_change_summary.csv",index=False,encoding="utf-8-sig")
object_summary_df.to_csv(output_folder/"informatica_xml_object_summary.csv",index=False,encoding="utf-8-sig")
with open(output_folder/"informatica_xml_detailed_changes.json","w",encoding="utf-8") as f:
    json.dump(detail_df.to_dict("records"),f,ensure_ascii=False,indent=2)
print(f"Reports written to {output_folder}")

## 10. Overall Result

In [ ]:
summary={
    "xml_files_old":len(old_files),
    "xml_files_new":len(new_files),
    "files_compared":len(all_keys),
    "files_changed":int((file_summary_df["status"]=="CHANGED").sum()) if not file_summary_df.empty else 0,
    "files_added":int((file_summary_df["status"]=="FILE_ADDED").sum()) if not file_summary_df.empty else 0,
    "files_removed":int((file_summary_df["status"]=="FILE_REMOVED").sum()) if not file_summary_df.empty else 0,
    "files_unchanged":int((file_summary_df["status"]=="UNCHANGED").sum()) if not file_summary_df.empty else 0,
    "detailed_changes":len(detail_df)
}
pd.DataFrame([summary])